# 📊 01 — Exploratory Data Analysis
## Marine Biodiversity Ecosystem Assessment
**Shri Harsan M | M.Tech Data Science | SRM Institute**

This notebook covers:
- Dataset composition (5,201 images, 3 species)
- Class distribution analysis
- Image quality statistics
- Bounding box size distribution
- Sample image visualisation

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
import pandas as pd
from collections import Counter

# Project root
ROOT = Path('..').resolve()
DATA = ROOT / 'dataset' / 'standardized'
print('Project root:', ROOT)
print('Dataset path:', DATA)

## 1. Dataset Split Overview

In [ ]:
CLASS_NAMES = {0: 'Butterflyfish', 1: 'Parrotfish', 2: 'Angelfish'}
CLASS_COLORS = {0: '#00ff9d', 1: '#8b5cf6', 2: '#00c8ff'}
CLASS_WEIGHTS = {0: 2.0, 1: 1.6, 2: 1.1}

splits = ['train', 'val', 'test']
stats = {}

for split in splits:
    img_dir = DATA / split / 'images'
    lbl_dir = DATA / split / 'labels'
    imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.jpeg')) + list(img_dir.glob('*.png'))
    
    class_counts = Counter()
    for lbl in lbl_dir.glob('*.txt'):
        with open(lbl) as f:
            for line in f:
                cls = int(line.split()[0])
                class_counts[cls] += 1
    
    stats[split] = {'images': len(imgs), 'labels': class_counts}
    print(f"{split:8s}: {len(imgs):4d} images | ", end='')
    for c, n in sorted(class_counts.items()):
        print(f"{CLASS_NAMES[c]}={n} ", end='')
    print()

## 2. Class Distribution Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Class Distribution Across Splits', fontsize=14, fontweight='bold')

for i, split in enumerate(splits):
    counts = [stats[split]['labels'].get(c, 0) for c in range(3)]
    colors = [CLASS_COLORS[c] for c in range(3)]
    bars = axes[i].bar(CLASS_NAMES.values(), counts, color=colors, alpha=0.85, edgecolor='white')
    axes[i].set_title(f'{split.capitalize()} Split', fontweight='bold')
    axes[i].set_xlabel('Species')
    axes[i].set_ylabel('Annotation Count')
    for bar, val in zip(bars, counts):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                     str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/preprocessing/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/preprocessing/class_distribution.png')

## 3. Bounding Box Size Distribution

In [ ]:
boxes_by_class = {0: [], 1: [], 2: []}

for split in splits:
    lbl_dir = DATA / split / 'labels'
    for lbl in lbl_dir.glob('*.txt'):
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, x, y, w, h = int(parts[0]), *map(float, parts[1:])
                    area = w * h * 100  # as % of image
                    boxes_by_class[cls].append(area)

fig, ax = plt.subplots(figsize=(10, 5))
for cls, areas in boxes_by_class.items():
    ax.hist(areas, bins=40, alpha=0.6, label=CLASS_NAMES[cls], color=CLASS_COLORS[cls])

ax.set_xlabel('Bounding Box Area (% of image)')
ax.set_ylabel('Frequency')
ax.set_title('Bounding Box Size Distribution by Species')
ax.legend()
plt.tight_layout()
plt.savefig('../results/preprocessing/bbox_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Sample Image Visualisation

In [ ]:
import random

def show_annotated(split, n=6):
    img_dir = DATA / split / 'images'
    lbl_dir = DATA / split / 'labels'
    imgs = list(img_dir.glob('*.jpg'))[:n]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'{split.capitalize()} Split — Sample Annotations', fontsize=13, fontweight='bold')
    
    for ax, img_path in zip(axes.flatten(), imgs):
        img = Image.open(img_path).convert('RGB')
        W, H = img.size
        ax.imshow(img)
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    cls, cx, cy, w, h = int(line.split()[0]), *map(float, line.split()[1:])
                    x1 = (cx - w/2) * W; y1 = (cy - h/2) * H
                    rect = patches.Rectangle((x1, y1), w*W, h*H,
                                             linewidth=2, edgecolor=CLASS_COLORS[cls], facecolor='none')
                    ax.add_patch(rect)
                    ax.text(x1, y1-4, CLASS_NAMES[cls], color=CLASS_COLORS[cls],
                            fontsize=8, fontweight='bold', backgroundcolor='black')
        ax.axis('off')
        ax.set_title(img_path.name[:20], fontsize=8)
    plt.tight_layout()
    plt.savefig(f'../results/preprocessing/samples_{split}.png', dpi=120, bbox_inches='tight')
    plt.show()

show_annotated('train')

## Summary

| Split | Images | Butterflyfish | Parrotfish | Angelfish |
|-------|--------|--------------|------------|----------|
| Train | ~3640  | ~1400        | ~1400      | ~840     |
| Val   | ~1197  | ~460         | ~460       | ~277     |
| Test  | 364    | 56           | 163        | 145      |
| **Total** | **5,201** | **~1916** | **~2023** | **~1262** |